# Week 6 Lecture: Control Flow, Iteration, and Functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-7025/blob/main/notebooks/tuesday-your-turn/week-06-lecture.ipynb)

This week you move from working with data to writing code that *thinks*. You'll learn how to make decisions with conditional statements, automate repetitive work with loops, and package reusable logic into functions. These three tools — control flow, iteration, and functions — are what separate one-off scripts from professional, maintainable analysis code.

## How to use this notebook

This notebook accompanies the Week 6 Tuesday lecture. You can:

- **Follow along during class** — run each cell as the instructor discusses it
- **Pause and experiment** — modify the code to test your understanding
- **Reference after class** — use it alongside the textbook chapters

## This Week's Topics

- **Chapter 16: Control Statements** — branch program logic with `if`, `elif`, and `else`; apply conditional logic across DataFrame columns with `np.where()`
- **Chapter 17: Iteration Statements** — automate repetitive tasks with `for` loops and write concise transformations with list comprehensions
- **Chapter 18: Writing Functions** — package reusable logic into clean, documented, testable functions and apply them to DataFrames

## Setup

Run this cell first. It loads all libraries and data used throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np
from completejourney_py import get_data

# Load the Complete Journey dataset — used throughout this notebook
cj = get_data()
transactions = cj['transactions']

print(f"Transactions loaded: {transactions.shape[0]:,} rows x {transactions.shape[1]} columns")
transactions.head(3)

---

## 1. Control Statements: Making Decisions in Code

Business logic almost always involves conditional decisions: if a customer qualifies for a discount, apply it; if inventory drops below a threshold, trigger an alert; if a sale exceeds a target, pay a bonus. Python's `if`, `elif`, and `else` statements let you encode those rules directly.

The key rule: Python checks conditions **top to bottom** and stops at the **first** `True` branch. That means your most specific condition must come first — a common ordering mistake will silently return wrong answers.

### Simple `if` — act only when a condition is met

In [ ]:
# Trigger a reorder alert when inventory falls below a threshold
stock_level = 5

if stock_level < 10:
    print("URGENT: Reorder needed!")

print(f"Current stock: {stock_level} units")

### Multi-branch `if / elif / else` — check conditions in order

In [ ]:
# Segment customers by annual spend and assign the right engagement strategy.
# Conditions are checked top-to-bottom — most restrictive threshold goes first.
annual_spend = 12500

if annual_spend >= 10000:
    segment = "High Value"
    strategy = "Personal account manager"
elif annual_spend >= 5000:
    segment = "Medium Value"
    strategy = "Quarterly check-ins"
elif annual_spend >= 1000:
    segment = "Low Value"
    strategy = "Email campaigns"
else:
    segment = "Inactive"
    strategy = "Re-engagement campaign"

print(f"Annual spend: ${annual_spend:,}")
print(f"Segment: {segment} → Strategy: {strategy}")

### Why order matters — a common mistake

In [ ]:
annual_spend = 12500

# WRONG — least specific condition checked first; stops at 'Low Value' every time
if annual_spend >= 1000:
    segment = "Low Value"    # always True for any spend >= 1000
elif annual_spend >= 5000:
    segment = "Medium Value" # never reached
elif annual_spend >= 10000:
    segment = "High Value"   # never reached
else:
    segment = "Inactive"

print(f"Wrong result: {segment}")  # prints 'Low Value' for a $12,500 spender!

### Dictionary as a switch pattern

For simple key-to-value lookups, a dictionary is faster and easier to maintain than a long `if/elif` chain. Adding a new option means adding one line to the dict — not rewriting conditional logic.

In [ ]:
# Look up shipping cost by method — dict is faster than 10 elif branches
shipping_costs = {
    'standard':    4.99,
    'express':     9.99,
    'overnight':  24.99,
    'pickup':      0.00,
}

method = 'express'

# .get() lets you supply a default if the key doesn't exist
cost = shipping_costs.get(method, 'Unknown method')
print(f"Shipping ({method}): ${cost}")

# Try an unknown method
print(shipping_costs.get('drone', 'Unknown method'))

### Try It — VIP Discount Logic

Write an `if/elif/else` block that assigns a `discount` based on `customer_type`:

- `'VIP'` → 20% discount (`0.20`)
- `'Premium'` → 15% discount (`0.15`)
- anything else → 5% discount (`0.05`)

Then print: `"VIP customer receives 20% discount"` (adjust for the actual values).

Test with `customer_type = 'VIP'`, `'Premium'`, and `'Regular'`.

In [ ]:
# Your code here


### Try It — Grocery Loyalty Classifier (from scratch)

Write code from scratch that assigns a loyalty status and discount based on `monthly_spend`:

| Monthly Spend | Status | Discount |
|--------------|--------|----------|
| ≥ $500 | Gold | 10% |
| ≥ $200 | Silver | 5% |
| ≥ $50 | Bronze | 2% |
| < $50 | No Status | 0% |

Print: `"$650/month → Gold (10% off)"`

Test with: $650, $275, $75, $30.

In [ ]:
# Your code here


---

## 2. Vectorized Conditionals: `np.where()`

A Python `if/else` block works on a single value. When you need to apply the same conditional logic to every row in a DataFrame, reach for `np.where()`. It processes thousands of rows in one vectorized operation — far faster than looping row by row.

**Syntax:** `np.where(condition, value_if_true, value_if_false)`

In [ ]:
# Apply a VIP discount to every row in one operation — no loop needed
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'is_vip':      [True, False, True, False, True],
    'base_price':  [100, 100, 100, 100, 100]
})

customers['final_price'] = np.where(
    customers['is_vip'],             # condition
    customers['base_price'] * 0.80,  # value when True  (VIP: 20% off)
    customers['base_price']          # value when False (standard price)
)

customers

In [ ]:
# Now on real data — flag every transaction in the Complete Journey dataset
transactions['value_tier'] = np.where(
    transactions['sales_value'] > 10,
    'premium',
    'standard'
)

# How many transactions fall into each tier?
transactions['value_tier'].value_counts()

### Try It — Flag Discounted Transactions

Use `np.where()` to add a new column `'disc_flag'` to the `transactions` DataFrame:

- `'discounted'` if `retail_disc > 0`
- `'full price'` otherwise

Then use `.value_counts()` to see the split. What share of transactions had a retail discount applied?

In [ ]:
# Your code here


---

## 3. Iteration: `for` Loops

A `for` loop lets Python repeat the same logic for every item in a sequence — a list, a dictionary, a range, or anything iterable. This is how you automate what would otherwise be tedious copy-paste: calculate commissions for every salesperson, read in files for every month, process every store in a region.

Three components: the **sequence** (what to iterate over), the **body** (what to do each time), and the **output** (print it, store it, or modify something).

In [ ]:
# Calculate commission for each sale and print the result
sales_amounts = [45000, 52000, 38000, 41000, 67000]

for sales in sales_amounts:
    commission = sales * 0.10
    print(f"Sales: ${sales:,} → Commission: ${commission:,.0f}")

In [ ]:
# Sometimes you want to collect results rather than print them
commissions = []

for sales in sales_amounts:
    commission = sales * 0.10
    commissions.append(commission)

print("All commissions:", commissions)
print(f"Total paid out: ${sum(commissions):,.0f}")

In [ ]:
# Iterate over a dictionary with .items() to get both key and value
store_data = {
    'Downtown': 125000,
    'Mall':      98000,
    'Suburbs':  142000,
    'Airport':   87000,
}

print(f"{'Store':<12} {'Sales':>10} {'Commission':>12}")
print("-" * 36)
for store_name, sales in store_data.items():
    commission = sales * 0.10
    print(f"{store_name:<12} ${sales:>9,} ${commission:>11,.0f}")

### Try It — Loyalty Classifier with a Loop

Earlier you wrote `if/elif/else` to classify a single household's loyalty status. Now use a `for` loop to classify all of them at once.

**Data:**
```python
households = [650, 275, 75, 30, 520, 190]
```

Loop over `households` and for each spending amount print:
`"$X/month → Status (Y% off)"`

Use the same Gold / Silver / Bronze / No Status rules from earlier.

> **The pattern:** put your `if/elif/else` logic *inside* the loop body — Python runs the full conditional for each item in the list.

In [ ]:
households = [650, 275, 75, 30, 520, 190]

# Your code here


---

## 4. List Comprehensions

List comprehensions are Python's shorthand for the build-and-append loop pattern. Instead of creating an empty list, looping, and calling `.append()`, you write the whole thing in one readable line.

**Syntax:** `[expression for item in sequence]`  
**With filter:** `[expression for item in sequence if condition]`  
**With conditional value:** `[value_if_true if condition else value_if_false for item in sequence]`

In [ ]:
prices = [29.99, 49.99, 19.99, 89.99, 120.00, 9.99]

# Traditional loop approach
discounted_loop = []
for price in prices:
    discounted_loop.append(price * 0.85)

# Comprehension — same result, one line
discounted_comp = [price * 0.85 for price in prices]

print("Loop result:       ", [round(p, 2) for p in discounted_loop])
print("Comprehension result:", [round(p, 2) for p in discounted_comp])

In [ ]:
# Filter: keep only prices above $25 after discount
eligible = [price * 0.85 for price in prices if price > 25]
print("Prices over $25 (discounted):", [round(p, 2) for p in eligible])

# Conditional value: discount items over $50, keep others at full price
tiered = [price * 0.80 if price > 50 else price for price in prices]
print("Tiered pricing:              ", [round(p, 2) for p in tiered])

# Dict comprehension — same idea with curly braces
price_map = {f"item_{i}": round(price * 0.85, 2) for i, price in enumerate(prices)}
print("\nDict comprehension:", price_map)

### Try It — Temperature Converter

You have 10 days of Celsius readings and need to convert them to Fahrenheit.

**Data:**
```python
celsius_temps = [20, 22, 19, 25, 23, 18, 21, 24, 20, 22]
```

**Formula:** `fahrenheit = (celsius * 9/5) + 32`

1. Use a **`for` loop** to convert each temperature and store results in `fahrenheit_loop`
2. Do the same with a **list comprehension** stored in `fahrenheit_comp`
3. Confirm both lists are equal

In [ ]:
celsius_temps = [20, 22, 19, 25, 23, 18, 21, 24, 20, 22]

# Part 1: for loop

# Part 2: list comprehension

# Part 3: confirm they match


---

## 5. Writing Functions

If you've copied and pasted the same block of code more than twice, it's time to write a function. Functions let you name a piece of logic, call it anywhere, and change it in one place when business rules evolve. They also make your code dramatically easier to test and share with teammates.

We'll build a profit margin function in four levels — each level adds something real without adding complexity for its own sake.

In [ ]:
# Level 1 — basic: inputs and an output
def calculate_profit_margin(revenue, cost):
    margin = (revenue - cost) / revenue
    return margin

print(calculate_profit_margin(1000, 600))   # 0.4 → 40% margin
print(calculate_profit_margin(500, 480))    # 0.04 → thin margin!

In [ ]:
# Level 2 — add a docstring so teammates know what this does
def calculate_profit_margin(revenue, cost):
    """
    Calculate gross profit margin as a decimal.

    Args:
        revenue: Total revenue from sales
        cost: Total cost of goods sold

    Returns:
        Profit margin as a decimal (e.g., 0.40 for 40%)
    """
    margin = (revenue - cost) / revenue
    return margin

# help() renders the docstring
help(calculate_profit_margin)

In [ ]:
# Level 3 — add type hints so IDEs and readers know what types to expect
def calculate_profit_margin(revenue: float, cost: float) -> float:
    """
    Calculate gross profit margin as a decimal.

    Args:
        revenue: Total revenue from sales
        cost: Total cost of goods sold

    Returns:
        Profit margin as a decimal (e.g., 0.40 for 40%)
    """
    margin = (revenue - cost) / revenue
    return margin

# Keyword arguments make the call self-documenting
m = calculate_profit_margin(revenue=1000, cost=600)
print(f"Margin: {m:.1%}")

In [ ]:
# Level 4 — add input validation so bad inputs fail loudly with clear messages
def calculate_profit_margin(revenue: float, cost: float) -> float:
    """
    Calculate gross profit margin as a decimal.

    Args:
        revenue: Total revenue from sales (must be > 0)
        cost: Total cost of goods sold (must be >= 0 and <= revenue)

    Returns:
        Profit margin as a decimal (e.g., 0.40 for 40%)
    """
    if revenue <= 0:
        raise ValueError("revenue must be greater than zero")
    if cost < 0:
        raise ValueError("cost cannot be negative")
    if cost > revenue:
        raise ValueError(f"cost ({cost}) exceeds revenue ({revenue})")

    return (revenue - cost) / revenue

# Normal use
print(f"{calculate_profit_margin(1000, 600):.1%}")  # 40.0%

# This will raise a clear ValueError
try:
    calculate_profit_margin(1000, 1200)
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# Functions shine when you need to apply the same calculation many times
def calculate_roi(revenue: float, cost: float) -> float:
    """Return on investment as a percentage."""
    return (revenue - cost) / cost * 100

campaigns = {
    'Email':   {'revenue': 15000, 'cost': 10000},
    'Social':  {'revenue':  8000, 'cost':  5000},
    'Search':  {'revenue': 12000, 'cost':  9000},
    'Display': {'revenue':  6000, 'cost':  4500},
}

# Dict comprehension + function = concise, readable reporting
roi_results = {
    name: calculate_roi(data['revenue'], data['cost'])
    for name, data in campaigns.items()
}

for channel, roi in sorted(roi_results.items(), key=lambda x: -x[1]):
    print(f"{channel:<10} {roi:>6.1f}% ROI")

### Try It — Customer Lifetime Value Function

Write a function `customer_lifetime_value()` that calculates CLV using the formula:

**CLV = avg_order_value × purchase_frequency × gross_margin × customer_lifespan**

Requirements:
- 4 parameters: `avg_order_value`, `purchase_frequency`, `gross_margin`, `customer_lifespan`
- Include a docstring
- Return the CLV value

**Test your function — expected results:**

| Call | Expected |
|------|----------|
| `(150, 4, 0.25, 3)` | `450.0` |
| `(200, 6, 0.30, 5)` | `1800.0` |
| `(100, 3, 0.20, 2)` | `120.0` |

In [ ]:
# Your code here


---

## 6. Functions + DataFrames

Functions become most powerful when combined with iteration or DataFrame operations. Once you've defined a function, you can apply it to every row of a DataFrame using `.apply()` — no loop needed. This is how professional analysts scale one-off calculations to thousands of customers, products, or transactions.

In [ ]:
# Define the CLV function (copy from your Try It if you completed it, or use this version)
def customer_lifetime_value(avg_order_value, purchase_frequency, gross_margin, customer_lifespan):
    """Calculate Customer Lifetime Value."""
    return round(avg_order_value * purchase_frequency * gross_margin * customer_lifespan, 2)

# Customer portfolio as a DataFrame
customers_df = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004'],
    'avg_order':   [150,    200,    100,    75],
    'frequency':   [4,      6,      3,      12],
    'margin':      [0.25,   0.30,   0.20,   0.15],
    'lifespan':    [3,      5,      2,      4],
})

# .apply() with a lambda bridges the DataFrame row to our function's parameters
customers_df['clv'] = customers_df.apply(
    lambda row: customer_lifetime_value(
        avg_order_value=row['avg_order'],
        purchase_frequency=row['frequency'],
        gross_margin=row['margin'],
        customer_lifespan=row['lifespan']
    ),
    axis=1  # apply across columns (row by row)
)

customers_df

In [ ]:
# Lambda functions are great for quick, inline transformations on real data
# Classify each transaction as 'high value' or 'low value' based on sales_value
transactions['purchase_size'] = transactions['sales_value'].apply(
    lambda x: 'high value' if x > 10 else 'low value'
)

# Summarize
transactions.groupby('purchase_size')['sales_value'].agg(['count', 'mean', 'sum']).round(2)

### Try It — Apply a Discount Function to a DataFrame

Write a function `apply_discount(price, category)` that returns a discounted price based on product category:

- `'produce'` → 15% off
- `'frozen'` → 10% off
- `'bakery'` → 20% off
- anything else → no discount

Then apply it to the DataFrame below using `.apply()` to add a `'discounted_price'` column.

In [ ]:
products_sample = pd.DataFrame({
    'product':  ['Apples', 'Frozen Pizza', 'Sourdough', 'Soda', 'Broccoli'],
    'category': ['produce', 'frozen', 'bakery', 'beverage', 'produce'],
    'price':    [3.99, 7.49, 5.29, 2.99, 2.49]
})

# Your code here


---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| `if / elif / else` | Check conditions top-to-bottom; most restrictive first |
| Dictionary switch | Faster and more maintainable than long `elif` chains for key-value lookups |
| `np.where()` | Apply if/else logic to an entire DataFrame column in one vectorized operation |
| `for` loop | Iterate over a sequence; put conditional logic *inside* the loop body |
| List comprehension | `[expression for item in sequence]` — concise alternative to build-and-append loops |
| Functions | Write once, reuse many times; add docstrings, type hints, and validation as complexity grows |
| `.apply()` | Bridge a custom function to every row of a DataFrame using a lambda |

## What's Next

- **Thursday Lab:** Project work session — bring your business question, your analysis plan, and your laptop. The goal is to leave with at least one complete analysis done.
- **Chapters to review:** [Chapter 16](../../book/16-control-statements.html), [Chapter 17](../../book/17-iteration-statements.html), [Chapter 18](../../book/18-functions.html) for deeper coverage and additional exercises
- **Cheat sheet:** [Module 6 Cheat Sheet](../../book/module-06-cheatsheet.html) for a quick reference on all of today's concepts